# Week 10 — Exploratory Data Analysis
UCI Online Shoppers Purchasing Intention (dataset 468). Run from the repository root after downloading the data.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd()
if not (ROOT / 'data').exists(): ROOT = ROOT.parent
DATA_PATH = ROOT / 'data' / 'raw' / 'online_shoppers.csv'
df = pd.read_csv(DATA_PATH)
print('Dataset shape:', df.shape)
display(df.head())
print('\nColumn data types:')
display(df.dtypes.to_frame('dtype'))

## Dataset structure and data quality
Check row/column counts, missing values, exact duplicates, and descriptive statistics. Exact duplicates are removed by the preprocessing pipeline before splitting.

In [ ]:
print('Missing values by column:')
display(df.isna().sum().sort_values(ascending=False).to_frame('missing_count'))
print('Total missing cells:', int(df.isna().sum().sum()))
print('Exact duplicate rows:', int(df.duplicated().sum()))
display(df.describe(include='all').T)

## Target distribution and class imbalance
`Revenue=True` is the minority purchase class. The primary metric is macro-F1; per-class recall is also important because accuracy alone can hide poor purchase detection.

In [ ]:
target_counts = df['Revenue'].value_counts(dropna=False).sort_index()
target_summary = pd.DataFrame({'count': target_counts, 'percentage': (target_counts / len(df) * 100).round(2)})
display(target_summary)
ax = target_counts.rename(index={False:'No purchase', True:'Purchase'}).plot(kind='bar', rot=0, title='Revenue class distribution')
ax.set_ylabel('Sessions'); plt.tight_layout(); plt.show()
print('Minority-class share:', f"{df['Revenue'].mean():.2%}")

## Numerical feature distributions
Histograms help identify skewed session-duration/page-value variables and unusual ranges. These plots are descriptive; no learned preprocessing is fitted on the full dataset.

In [ ]:
numeric = df.select_dtypes(include='number')
numeric.hist(figsize=(15, 11), bins=30)
plt.suptitle('Numerical feature distributions')
plt.tight_layout(); plt.show()

## Correlation inspection
The correlation matrix is descriptive and applies only to numeric columns. Categorical variables are encoded later inside a training-fitted preprocessing pipeline.

In [ ]:
corr = numeric.corr(numeric_only=True)
display(corr.round(2))
display(corr['Revenue'].sort_values(ascending=False).to_frame('correlation_with_Revenue'))

## Week 10 findings and modelling implications
- Report the observed class counts and percentages from the executed cells; do not rely on approximate percentages alone.
- Compare a majority-class DummyClassifier with logistic regression.
- Compare unweighted and class-weighted logistic regression on the same validation set; discuss both purchase and no-purchase recall.
- Fit imputers, StandardScaler, and OneHotEncoder on training data only. Validation and test data are transform-only.
- Keep the test split untouched until the final evaluation after model selection.